# Demo 1 — Phoenix Datasets & Experiments

Shows the full Phoenix evaluation workflow end-to-end:

1. **Dataset** — upload a versioned set of tax Q&A examples to Phoenix
2. **Experiment** — run the RAG pipeline on every example, log outputs
3. **Evaluations** — score each run with LLM-as-judge (faithfulness + correctness)
4. **UI** — view dataset, experiment runs, and eval scores side-by-side in Phoenix

Everything uses the Phoenix REST API (`http://localhost:6006`) — no extra packages needed beyond what's already installed.

**Prerequisites:** `docker compose up -d` running, `01_ingest_and_embed.ipynb` already run.

In [7]:
import os, sys, json, time, requests
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv('../.env')

PHOENIX_BASE_URL  = os.getenv('PHOENIX_BASE_URL',  'http://localhost:6006')
OLLAMA_BASE_URL   = os.getenv('OLLAMA_BASE_URL',   'http://localhost:11434')
OLLAMA_LLM_MODEL  = os.getenv('OLLAMA_LLM_MODEL',  'llama3.2')

sys.path.insert(0, '..')

# Verify Phoenix is reachable
r = requests.get(f'{PHOENIX_BASE_URL}/v1/datasets')
print(f'Phoenix {PHOENIX_BASE_URL} → HTTP {r.status_code}')

Phoenix http://localhost:6006 → HTTP 200


## 1. Create a Dataset in Phoenix

A Phoenix **dataset** is a versioned collection of input/output examples.
We upload 5 tax Q&A pairs. Each example has:
- `input.question` — the user question
- `output.reference_answer` — the known correct answer
- `metadata` — tags for filtering

Datasets are immutable once created — each upload creates a new version.

In [8]:
DATASET_NAME = 'tax-rag-eval-v1'

examples = [
    {
        'question': 'What is the standard deduction for a single filer in 2024?',
        'reference_answer': '$14,600',
        'topic': 'standard_deduction',
    },
    {
        'question': 'How are long-term capital gains taxed for someone earning $100,000?',
        'reference_answer': '15% rate applies for most middle-income earners',
        'topic': 'capital_gains',
    },
    {
        'question': 'What is the 401(k) contribution limit for 2024 including catch-up?',
        'reference_answer': '$30,500 total with catch-up for age 50+',
        'topic': '401k',
    },
    {
        'question': 'Can I deduct student loan interest if I earn $80,000 as a single filer?',
        'reference_answer': 'Yes, up to $2,500 but phases out between $75,000–$90,000 MAGI',
        'topic': 'student_loan',
    },
    {
        'question': 'What are the Roth IRA income phase-out ranges for single filers in 2024?',
        'reference_answer': '$146,000–$161,000 for single filers',
        'topic': 'roth_ira',
    },
]

resp = requests.post(
    f'{PHOENIX_BASE_URL}/v1/datasets/upload',
    json={
        'action': 'create',
        'name': DATASET_NAME,
        'description': 'Tax policy RAG golden evaluation set — 5 questions with reference answers',
        'inputs':   [{'question': e['question']}               for e in examples],
        'outputs':  [{'reference_answer': e['reference_answer']} for e in examples],
        'metadata': [{'topic': e['topic']}                     for e in examples],
    },
)

if resp.status_code == 409:
    # Dataset already exists — fetch the existing one by name
    all_datasets = requests.get(f'{PHOENIX_BASE_URL}/v1/datasets').json()['data']
    existing = next(d for d in all_datasets if d['name'] == DATASET_NAME)
    dataset_id = existing['id']
    print(f'Dataset already exists  id={dataset_id}')
else:
    resp.raise_for_status()
    dataset_id = resp.json()['data']['dataset_id']
    print(f'Dataset created  id={dataset_id}')

print(f'Open → {PHOENIX_BASE_URL}/datasets')

Dataset already exists  id=RGF0YXNldDox
Open → http://localhost:6006/datasets


In [9]:
import pandas as pd

# Fetch examples back to get their Phoenix-assigned IDs (needed for experiment runs)
resp = requests.get(f'{PHOENIX_BASE_URL}/v1/datasets/{dataset_id}/examples')
resp.raise_for_status()
raw_examples = resp.json()['data']['examples']

examples_df = pd.DataFrame([
    {
        'example_id':       ex['id'],
        'question':         ex['input']['question'],
        'reference_answer': ex['output']['reference_answer'],
        'topic':            ex['metadata']['topic'],
    }
    for ex in raw_examples
])

print(f'Fetched {len(examples_df)} examples')
examples_df

Fetched 5 examples


,example_id,question,reference_answer,topic
0,RGF0YXNldEV4YW1wbGU6MQ==,What is the standard deduction for a single fi...,"$14,600",standard_deduction
1,RGF0YXNldEV4YW1wbGU6Mg==,How are long-term capital gains taxed for some...,15% rate applies for most middle-income earners,capital_gains
2,RGF0YXNldEV4YW1wbGU6Mw==,What is the 401(k) contribution limit for 2024...,"$30,500 total with catch-up for age 50+",401k
3,RGF0YXNldEV4YW1wbGU6NA==,Can I deduct student loan interest if I earn $...,"Yes, up to $2,500 but phases out between $75,0...",student_loan
4,RGF0YXNldEV4YW1wbGU6NQ==,What are the Roth IRA income phase-out ranges ...,"$146,000–$161,000 for single filers",roth_ira


## 2. Create an Experiment

A Phoenix **experiment** runs a task over every example in a dataset and records the outputs.
The experiment is linked to the dataset — Phoenix shows both side-by-side in the UI.

We create an experiment called `llama32-rag-run-1` for the RAG pipeline using llama3.2.

In [10]:
resp = requests.post(
    f'{PHOENIX_BASE_URL}/v1/datasets/{dataset_id}/experiments',
    json={
        'name':        f'llama32-rag-run-1',
        'description': f'RAG pipeline using {OLLAMA_LLM_MODEL} — retrieval k=4',
        'metadata':    {'model': OLLAMA_LLM_MODEL, 'retrieval_k': 4},
    },
)
resp.raise_for_status()
experiment_id = resp.json()['data']['id']
print(f'Experiment created  id={experiment_id}')
print(f'Open → {PHOENIX_BASE_URL}/datasets/{dataset_id}/experiments/{experiment_id}')

Experiment created  id=RXhwZXJpbWVudDoy
Open → http://localhost:6006/datasets/RGF0YXNldDox/experiments/RXhwZXJpbWVudDoy


## 3. Run the Task — RAG Pipeline on Each Example

For each dataset example:
1. Run `query_rag(question)` — this also sends a trace to Phoenix automatically
2. POST the output + timing to Phoenix as an **experiment run**

Phoenix records the task output alongside the dataset input/reference for comparison.

In [11]:
from app.rag_pipeline import query_rag

run_records = []  # collect for evaluation step

for _, row in examples_df.iterrows():
    print(f'Running: {row["question"][:60]}...')
    t0 = datetime.now(timezone.utc)
    result = query_rag(row['question'])
    t1 = datetime.now(timezone.utc)

    run_payload = {
        'dataset_example_id': row['example_id'],
        'repetition_number':  1,
        'output': {
            'answer':   result['answer'],
            'sources':  result['sources'],
            'chunks':   result['retrieved_chunks'],
        },
        'start_time': t0.isoformat(),
        'end_time':   t1.isoformat(),
    }
    r = requests.post(
        f'{PHOENIX_BASE_URL}/v1/experiments/{experiment_id}/runs',
        json=run_payload,
    )
    r.raise_for_status()
    run_id = r.json()['data']['id']

    run_records.append({
        'run_id':           run_id,
        'example_id':       row['example_id'],
        'question':         row['question'],
        'reference_answer': row['reference_answer'],
        'context':          '\n\n'.join(result.get('context_chunks', [])),
        'answer':           result['answer'],
        'latency_s':        (t1 - t0).total_seconds(),
    })
    print(f'  → run_id={run_id}  latency={run_records[-1]["latency_s"]:.1f}s')

import pandas as pd
runs_df = pd.DataFrame(run_records)
print(f'\nAll {len(runs_df)} runs logged.')
runs_df[['question', 'answer', 'latency_s']]

Running: What is the standard deduction for a single filer in 2024?...
  → run_id=RXhwZXJpbWVudFJ1bjo2  latency=6.5s
Running: How are long-term capital gains taxed for someone earning $1...
  → run_id=RXhwZXJpbWVudFJ1bjo3  latency=15.2s
Running: What is the 401(k) contribution limit for 2024 including cat...
  → run_id=RXhwZXJpbWVudFJ1bjo4  latency=10.8s
Running: Can I deduct student loan interest if I earn $80,000 as a si...
  → run_id=RXhwZXJpbWVudFJ1bjo5  latency=17.0s
Running: What are the Roth IRA income phase-out ranges for single fil...
  → run_id=RXhwZXJpbWVudFJ1bjoxMA==  latency=11.2s

All 5 runs logged.


,question,answer,latency_s
0,What is the standard deduction for a single fi...,The standard deduction amount for a single fil...,6.483449
1,How are long-term capital gains taxed for some...,"According to the context excerpt, the tax rate...",15.197579
2,What is the 401(k) contribution limit for 2024...,"According to the context excerpt, the 401(k), ...",10.785984
3,Can I deduct student loan interest if I earn $...,"According to the provided context excerpt, yes...",17.016332
4,What are the Roth IRA income phase-out ranges ...,"Unfortunately, I couldn't find any specific in...",11.158386


## 4. Evaluate Each Run — LLM-as-Judge

Run two evaluators (faithfulness + correctness) on every experiment run,
then POST each result back to Phoenix as an **experiment evaluation**.

Phoenix shows eval scores inline with the experiment run table in the UI.

In [12]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat_model = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL)

FAITHFULNESS_PROMPT = """\
You are evaluating whether an AI answer is grounded in the provided context.

Context: {context}
Answer: {output}

Is the answer faithful to the context (only contains information supported by the context)?
Respond with exactly one word: faithful or unfaithful"""

CORRECTNESS_PROMPT = """\
You are evaluating whether an AI answer is correct compared to a reference answer.

Question: {input}
Reference Answer: {reference}
AI Answer: {output}

Is the AI answer semantically correct compared to the reference?
Respond with exactly one word: correct or incorrect"""

def _label(response, positive, negative):
    text = response.content.strip().lower()
    return negative if negative in text else positive

eval_df = runs_df.rename(columns={
    'question':         'input',
    'context':          'context',
    'answer':           'output',
    'reference_answer': 'reference',
})

print('Running faithfulness evaluations...')
faith_labels = [
    _label(
        chat_model.invoke([HumanMessage(content=FAITHFULNESS_PROMPT.format(
            context=row['context'], output=row['output']
        ))]),
        'faithful', 'unfaithful',
    )
    for _, row in eval_df.iterrows()
]

print('Running correctness evaluations...')
correct_labels = [
    _label(
        chat_model.invoke([HumanMessage(content=CORRECTNESS_PROMPT.format(
            input=row['input'], reference=row['reference'], output=row['output']
        ))]),
        'correct', 'incorrect',
    )
    for _, row in eval_df.iterrows()
]

faith_results   = pd.DataFrame({'label': faith_labels})
correct_results = pd.DataFrame({'label': correct_labels})

print('Evaluations complete.')
print(faith_results['label'].value_counts().to_string())
print(correct_results['label'].value_counts().to_string())

Running faithfulness evaluations...
Running correctness evaluations...
Evaluations complete.
label
unfaithful    4
faithful      1
label
correct      3
incorrect    2


## 5. Upload Evaluation Results to Phoenix

POST each evaluation back to Phoenix via `POST /v1/experiment_evaluations`.
After this, the Phoenix **Experiments** tab shows the eval scores alongside each run.

In [13]:
LABEL_TO_SCORE = {
    'faithful': 1.0, 'unfaithful': 0.0,
    'correct': 1.0, 'incorrect': 0.0,
}

def post_evaluation(run_id, eval_name, label, explanation=None):
    now = datetime.now(timezone.utc).isoformat()
    payload = {
        'experiment_run_id': run_id,
        'name':              eval_name,
        'annotator_kind':    'LLM',
        'start_time':        now,
        'end_time':          now,
        'result': {
            'label':       label,
            'score':       LABEL_TO_SCORE.get(label),
            'explanation': explanation,
        },
    }
    r = requests.post(f'{PHOENIX_BASE_URL}/v1/experiment_evaluations', json=payload)
    r.raise_for_status()
    return r.json()

for i, row in runs_df.iterrows():
    run_id = row['run_id']

    faith_label = faith_results.iloc[i]['label']
    faith_expl  = faith_results.iloc[i].get('explanation')
    post_evaluation(run_id, 'faithfulness', faith_label, faith_expl)

    corr_label  = correct_results.iloc[i]['label']
    corr_expl   = correct_results.iloc[i].get('explanation')
    post_evaluation(run_id, 'correctness', corr_label, corr_expl)

    print(f'Run {i+1}: faithfulness={faith_label}  correctness={corr_label}')

print(f'\nAll evaluations uploaded.')
print(f'Open Phoenix → {PHOENIX_BASE_URL}/datasets')

Run 1: faithfulness=unfaithful  correctness=correct
Run 2: faithfulness=faithful  correctness=correct
Run 3: faithfulness=unfaithful  correctness=incorrect
Run 4: faithfulness=unfaithful  correctness=correct
Run 5: faithfulness=unfaithful  correctness=incorrect

All evaluations uploaded.
Open Phoenix → http://localhost:6006/datasets


## 6. Summary

What just happened and where to look in the Phoenix UI:

In [14]:
summary = runs_df[['question', 'answer', 'latency_s']].copy()
summary['faithful'] = faith_results['label'].values
summary['correct']  = correct_results['label'].values

print('=' * 60)
print('Experiment Summary')
print('=' * 60)
print(f'Dataset       : {DATASET_NAME}  ({len(runs_df)} examples)')
print(f'Model         : {OLLAMA_LLM_MODEL}')
print(f'Faithful      : {(summary["faithful"] == "faithful").mean():.0%}')
print(f'Correct       : {(summary["correct"] == "correct").mean():.0%}')
print(f'Avg latency   : {summary["latency_s"].mean():.1f}s')
print()
print('Phoenix UI:')
print(f'  Datasets    → {PHOENIX_BASE_URL}/datasets')
print(f'  Experiment  → {PHOENIX_BASE_URL}/datasets/{dataset_id}/experiments/{experiment_id}')
print()
summary

Experiment Summary
Dataset       : tax-rag-eval-v1  (5 examples)
Model         : llama3.2
Faithful      : 20%
Correct       : 60%
Avg latency   : 12.1s

Phoenix UI:
  Datasets    → http://localhost:6006/datasets
  Experiment  → http://localhost:6006/datasets/RGF0YXNldDox/experiments/RXhwZXJpbWVudDoy



,question,answer,latency_s,faithful,correct
0,What is the standard deduction for a single fi...,The standard deduction amount for a single fil...,6.483449,unfaithful,correct
1,How are long-term capital gains taxed for some...,"According to the context excerpt, the tax rate...",15.197579,faithful,correct
2,What is the 401(k) contribution limit for 2024...,"According to the context excerpt, the 401(k), ...",10.785984,unfaithful,incorrect
3,Can I deduct student loan interest if I earn $...,"According to the provided context excerpt, yes...",17.016332,unfaithful,correct
4,What are the Roth IRA income phase-out ranges ...,"Unfortunately, I couldn't find any specific in...",11.158386,unfaithful,incorrect


## What to Show in the Phoenix UI

After running all cells, open http://localhost:6006:

| Phoenix Tab | What you see |
|---|---|
| **Datasets** | `tax-rag-eval-v1` dataset with 5 examples, input/output/metadata columns |
| **Dataset → Experiments** | `llama32-rag-run-1` experiment linked to the dataset |
| **Experiment runs table** | Each row = one Q&A run with `faithfulness` and `correctness` scores inline |
| **Click a run** | Side-by-side: dataset input vs RAG output vs reference answer |
| **Traces** | Each `query_rag()` call also appears as a trace in the `tax-rag-demo` project |